In [15]:
# add parent to path
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from pathlib import Path

from wandb_utils import combine_histories, get_wandb_stats

# tpc-h bespoke storage lineage
run_lineages = {
    "tpch": [
        "nhpul25g",  # plan storage
        "a2tlnfrk",  # basic impl
        "h1mv9o9r",  # optim
    ],
    "ceb": [
        "x527bk9j",  # plan storage
        "blqeh6i0",  # basic impl
        "5qumtphx",  # optim
    ],
}

# run_id = "k9xst454"  # tpch - bespoke storage
# run_id = "ggikqms8"  # ceb
# run_id = "761bg8oe"  # ceb - bespoke storage

combined_hist_dict = dict()
for name, run_ids in run_lineages.items():
    hists_list = []
    for run_id in run_ids:
        summary, hist, config = get_wandb_stats(
            run_id,
            skip_cache=False,  # set to True to skip cache and fetch from W&B API
            wandb_run_cache_path=Path("/mnt/labstore/bespoke_olap/wandb_cache"),
        )
        hists_list.append(hist)

    combined_history = combine_histories(hists_list)

    # rewrite type if validation/external_call=True to validate_external
    combined_history["type"] = combined_history.apply(
        lambda row: (
            "validate_external"
            if row["validation/external_call"] == True
            else row["type"]
        ),
        axis=1,
    )

    combined_hist_dict[name] = combined_history

Loaded wandb data from cache: /mnt/labstore/bespoke_olap/wandb_cache/f813798a06f5f88ec7274e5087b4eefca156eb387a94fda7125264bafec6d3be.pkl
Loaded wandb data from cache: /mnt/labstore/bespoke_olap/wandb_cache/a88e90a90e889bfc1d66cfbd9ad3a7d47ac9133ece44f026d8deaf5502167226.pkl
Loaded wandb data from cache: /mnt/labstore/bespoke_olap/wandb_cache/a34a901d2f50cc51db2092712656d83dfe9b6d7757785b785003b489b84435c6.pkl
Combined history has 5672 rows ([7, 1001, 4664])
Loaded wandb data from cache: /mnt/labstore/bespoke_olap/wandb_cache/50bb348702d57dcdd091040e47023a2aa7fe4b9801a557e75c93077ff2b4ea6c.pkl
Loaded wandb data from cache: /mnt/labstore/bespoke_olap/wandb_cache/1dbfc6229436985d1a881d52fac106ad1e819507d67291515fd539e97955e38b.pkl
Loaded wandb data from cache: /mnt/labstore/bespoke_olap/wandb_cache/3c6b188c073e75be45f0039e047104d01e92bf0dc2890ac217df99662b9270d3.pkl
Combined history has 4954 rows ([7, 839, 4108])


# Compile Tool Stats

In [16]:
def get_compile_stats(combined_history, verbose: bool = False):
    # count compile invocations
    # considering only internal managed compile calls for now, as external calls are often just benchmarking

    filtered_hist = combined_history[
        combined_history["type"].isin(["compile", "validate"])
    ]

    filtered_tool_counts = filtered_hist["type"].value_counts().to_dict()

    compile_count = filtered_tool_counts.get("compile", 0)
    validate_count = filtered_tool_counts.get("validate", 0)

    # count errors
    compile_err_col = "compile/compile_error"
    if compile_err_col in filtered_hist.columns:
        compile_error_ctr = (
            filtered_hist[compile_err_col].value_counts().to_dict().get(True, 0)
        )
    else:
        compile_error_ctr = 0
    validate_compile_error_ctr = (
        filtered_hist["validation/compile_error"].value_counts().to_dict().get(True, 0)
    )

    # totals
    total_count = compile_count + validate_count
    total_errors = compile_error_ctr + validate_compile_error_ctr

    if verbose:
        print(
            f"Total compile-related tool invocations: {total_count}, with {total_errors} errors (error rate: {total_errors / total_count * 100 if total_count > 0 else 0:.2f}%) - compile only: {compile_count} (errors: {compile_error_ctr})."
        )

    return {
        "count": total_count,
        "error_rate": total_errors / total_count if total_count > 0 else 0,
    }


get_compile_stats(combined_hist_dict["tpch"], verbose=True)

Total compile-related tool invocations: 476, with 9 errors (error rate: 1.89%) - compile only: 0 (errors: 0).


{'count': 476, 'error_rate': 0.018907563025210083}

# Shell Tool Stats

In [17]:
import shlex


def _cmd_head(cmd: str) -> str:
    if not isinstance(cmd, str):
        return ""
    cmd = cmd.strip()
    if not cmd:
        return ""
    try:
        parts = shlex.split(cmd)
    except Exception:
        parts = cmd.split()
    return parts[0].lower() if parts else ""


def _has_any_token(cmd: str, tokens: set[str]) -> bool:
    if not isinstance(cmd, str):
        return False
    try:
        parts = [p.lower() for p in shlex.split(cmd)]
    except Exception:
        parts = cmd.lower().split()
    return any(t in parts for t in tokens)


def get_shell_tool_stats(combined_history, verbose: bool = False):
    # get shell tool stats
    shell_calls = combined_history[combined_history["type"] == "shell_command"]

    num_shell_tool_calls = shell_calls.shape[0]

    # flatten shell cmd list: shell/commands
    shell_cmds = shell_calls["shell/commands"].explode()
    # classify shell cmds by intent (more semantic than raw startswith)
    classes = {
        "filesystem (ls, find, ...)": lambda cmd: (
            _cmd_head(cmd) in {"ls", "find", "tree", "pwd", "stat", "du", "file"}
        ),
        "file reading (cat, ...)": lambda cmd: (
            _cmd_head(cmd) in {"cat", "head", "tail", "less", "more", "bat"}
        ),
        "text processing/search (grep, ...)": lambda cmd: (
            _cmd_head(cmd)
            in {"grep", "rg", "ag", "sed", "awk", "cut", "sort", "uniq", "wc", "nl"}
        ),
        "code execution (python, ...)": lambda cmd: (
            _cmd_head(cmd)
            in {"python", "python3", "ipython", "bash", "sh", "zsh", "make", "cmake"}
        ),
        "parquet analysis": lambda cmd: (
            _cmd_head(cmd) in {"parquet-tools", "parquet-dump-schema", "jq"}
        ),
        "environment/cmd lookup (which, ...)": lambda cmd: (
            _cmd_head(cmd) in {"which", "whereis", "type", "env", "printenv", "git"}
        ),
        "file mutation (cp, mv, ...)": lambda cmd: (
            _cmd_head(cmd)
            in {"cp", "mv", "rm", "mkdir", "rmdir", "touch", "chmod", "chown"}
        ),
        "output/formatting (echo, printf)": lambda cmd: (
            _cmd_head(cmd) in {"echo", "printf"}
        ),
        "pipeline-heavy": lambda cmd: _has_any_token(cmd, {"|", "xargs"}),
    }

    shell_cmd_classes = shell_cmds.apply(
        lambda cmd: next((cls for cls, fn in classes.items() if fn(cmd)), cmd)
    )

    if verbose:
        print(shell_cmd_classes.value_counts().to_dict())

    return {
        "count": num_shell_tool_calls,
        "shell_cmd_classes": shell_cmd_classes.value_counts().to_dict(),
    }


get_shell_tool_stats(combined_hist_dict["tpch"], verbose=True)

{'text processing/search (grep, ...)': 1724, 'file reading (cat, ...)': 319, 'filesystem (ls, find, ...)': 135, 'environment/cmd lookup (which, ...)': 9, 'code execution (python, ...)': 7, 'parquet analysis': 2, 'file mutation (cp, mv, ...)': 1}


{'count': 1466,
 'shell_cmd_classes': {'text processing/search (grep, ...)': 1724,
  'file reading (cat, ...)': 319,
  'filesystem (ls, find, ...)': 135,
  'environment/cmd lookup (which, ...)': 9,
  'code execution (python, ...)': 7,
  'parquet analysis': 2,
  'file mutation (cp, mv, ...)': 1}}

# Apply Patch Stats

In [18]:
def get_apply_patch_stats(combined_history, verbose: bool = False):
    filtered_hist = combined_history[(combined_history["type"] == "apply_patch_tool")]

    num_apply_patch_calls = len(filtered_hist)

    # compute avg between added and deleted loc count
    diff_size = (
        filtered_hist[["apply_patch/added_loc_count", "apply_patch/deleted_loc_count"]]
        .fillna(0)
        .max(axis=1)
    )

    # compute avg diff size
    avg_diff_size = float(diff_size.mean())

    if verbose:
        print(
            f"Total apply_patch_tool calls: {num_apply_patch_calls}, with average diff size (added or deleted LOC): {avg_diff_size:.2f}."
        )

    return {
        "count": num_apply_patch_calls,
        "avg_diff_size": avg_diff_size,
    }


get_apply_patch_stats(combined_hist_dict["tpch"], verbose=True)

Total apply_patch_tool calls: 616, with average diff size (added or deleted LOC): 52.70.


{'count': 616, 'avg_diff_size': 52.69642857142857}

# Validate Call

In [19]:
def get_validate_stats(combined_history, verbose: bool = False):
    # count compile invocations
    # considering only internal managed compile calls for now, as external calls are often just benchmarking

    type_mask = combined_history["type"].isin(["validate", "validate_external"])
    no_compile_error_mask = (
        combined_history["validation/compile_error"].fillna(False).eq(False)
    )  # keep rows where compile_error is False or NaN

    filtered_hist = combined_history[type_mask & no_compile_error_mask]

    validate_count = filtered_hist[filtered_hist["type"] == "validate"].shape[0]
    validate_external_count = filtered_hist[
        filtered_hist["type"] == "validate_external"
    ].shape[0]

    # count errors
    incorrect_elems = filtered_hist[filtered_hist["validation/correct"] == False]
    validate_errors = incorrect_elems[incorrect_elems["type"] == "validate"].shape[0]
    validate_errors_external = incorrect_elems[
        incorrect_elems["type"] == "validate_external"
    ].shape[0]

    # compute error rate internal
    error_rate_internal = validate_errors / validate_count if validate_count > 0 else 0
    error_rate_external = (
        validate_errors_external / validate_external_count
        if validate_external_count > 0
        else 0
    )

    return {
        "count": validate_count,
        "error_rate_internal": error_rate_internal,
        "validate_external_count": validate_external_count,
        "error_rate_external": error_rate_external,
    }


get_validate_stats(combined_hist_dict["tpch"], verbose=True)

{'count': 467,
 'error_rate_internal': 0.014989293361884369,
 'validate_external_count': 294,
 'error_rate_external': 0.003401360544217687}

# Assemble Output

In [20]:
rows = []
shell_types = dict()

TOOL_TYPES = {
    "shell_command",
    "apply_patch_tool",
    "compile",
    "validate",
    "validate_external",
}

for name, combined_history in combined_hist_dict.items():
    compile_stats = get_compile_stats(combined_history=combined_history)
    shell_stats = get_shell_tool_stats(combined_history=combined_history)
    apply_patch_stats = get_apply_patch_stats(combined_history=combined_history)
    validate_stats = get_validate_stats(combined_history=combined_history)

    tool_type_mask = combined_history["type"].isin(TOOL_TYPES)
    tool_count = int(tool_type_mask.sum())
    llm_count = int((~tool_type_mask).sum())
    total_interactions = int(len(combined_history))
    llm_share = llm_count / total_interactions if total_interactions > 0 else 0

    row = [
        name,
        f"{total_interactions}",
        f"{llm_count} ({llm_share * 100:.2f}%)",
        f"{shell_stats['count']}",
        f"{apply_patch_stats['count']} (avg diff size: {apply_patch_stats['avg_diff_size']:.2f} LOC)",
        f"{compile_stats['count']} (errors: {compile_stats['error_rate'] * 100:.2f}%)",
        f"{validate_stats['count']} (errors: {validate_stats['error_rate_internal'] * 100:.2f}%)\n + {validate_stats['validate_external_count']} framework invocations",
        f"{tool_count}",
    ]
    rows.append(row)

    shell_types[name] = shell_stats["shell_cmd_classes"]

# print tabulate
from collections import defaultdict

from tabulate import tabulate

print(
    tabulate(
        rows,
        headers=[
            "Name",
            "Total Interactions",
            "LLM Interactions",
            "Shell Tool",
            "Apply Patch",
            "Compile",
            "Validate",
            "Tool Calls",
        ],
    )
)


# print shell tool types distribution
total_type_count = defaultdict(int)
for name, shell_cmd_classes in shell_types.items():
    for cls, count in shell_cmd_classes.items():
        total_type_count[cls] += count

# print relative
total_calls = sum(total_type_count.values())
dist_rows = [
    [cls, count, f"{(count / total_calls * 100) if total_calls else 0:.2f}%"]
    for cls, count in total_type_count.items()
]

print(
    tabulate(
        dist_rows,
        headers=["Shell Command Type", "Count", "Share"],
    )
)

# print llm interaction type distribution
total_llm_type_count = defaultdict(int)
for combined_history in combined_hist_dict.values():
    llm_hist = combined_history[~combined_history["type"].isin(TOOL_TYPES)]
    for llm_type, count in llm_hist["type"].value_counts().to_dict().items():
        total_llm_type_count[llm_type] += int(count)

llm_total_calls = sum(total_llm_type_count.values())
llm_dist_rows = [
    [
        llm_type,
        count,
        f"{(count / llm_total_calls * 100) if llm_total_calls else 0:.2f}%",
    ]
    for llm_type, count in total_llm_type_count.items()
]

print(
    tabulate(
        llm_dist_rows,
        headers=["LLM Interaction Type", "Count", "Share"],
    )
)

Name      Total Interactions  LLM Interactions      Shell Tool  Apply Patch                     Compile              Validate                        Tool Calls
------  --------------------  ------------------  ------------  ------------------------------  -------------------  ----------------------------  ------------
tpch                    5672  2820 (49.72%)               1466  616 (avg diff size: 52.70 LOC)  476 (errors: 1.89%)  467 (errors: 1.50%)                   2852
                                                                                                                      + 294 framework invocations
ceb                     4954  2463 (49.72%)               1338  541 (avg diff size: 84.65 LOC)  389 (errors: 5.66%)  362 (errors: 10.22%)                  2491
                                                                                                                      + 208 framework invocations
Shell Command Type                     Count  Share
----------------